# Case 8 -- Two-Fluids, Two Stress periods, 1 layer



In [ ]:
from IPython.display import HTML
import pathlib as pl
import numpy as np
import matplotlib.pyplot as plt
import flopy
from swiutil import SwiAnimator


# path to mf6 executables with swi support: 
#   https://github.com/christianlangevin/modflow6-nightly-build/actions/workflows/nightly-build-swi.yml

# Put the name of the mf6 executable into mf6exe.txt,
# which is not under version control.
with open(pl.Path("./mf6exe.txt"), "r") as f:
    mf6exe = f.readline().strip()
print(f"using executable: {mf6exe}")

sim_ws = pl.Path("./temp/case8")

In [ ]:
is_confined = False

ncol = 19 + 2
Lx = 1900.0
dx = Lx / (ncol - 2)
nlay = 1
nrow = 1
delr = np.array([1.] + (ncol - 2) * [dx] + [1.])
#delr = np.array(ncol * [dx])


delc = 1.0
botm = -80.0
recharge = {0: 0.0075, 1: 0.0}
k_fw = 10.0
k_sw = 10.0 #9.403669797
h0 = 0.0

icelltype = 1
iconvert = 1
top = 10.0
if is_confined:
    icelltype = 0
    iconvert = 0
    top = 0.0

newtonoptions = "NEWTON"
ss = 0.0
sy = 0.2
inner_dvclose = 1.e-8
outer_dvclose = 1.e-7
# inner_dvclose = 1.e-4
# outer_dvclose = 1.e-3

perioddata = [(15000., 100, 1.0), (15000., 100, 1.0)] # this runs


def build_gwf_model(sim, is_saltwater):
    if is_saltwater:
        name = "saltwater"
    else:
        name = "freshwater"

    gwf = flopy.mf6.ModflowGwf(
        sim,
        modelname=name,
        save_flows=True,
        newtonoptions=newtonoptions,
    )
    dis = flopy.mf6.ModflowGwfdis(
        gwf,
        nlay=nlay,
        nrow=nrow,
        ncol=ncol,
        delr=delr,
        delc=delc,
        top=top,
        botm=botm,
    )
    strt = 0.0 if is_saltwater else 0.001
    ic = flopy.mf6.ModflowGwfic(gwf, strt=strt)
    npf = flopy.mf6.ModflowGwfnpf(
        gwf,
        save_specific_discharge=True,
        save_saturation=True,
        # alternative_cell_averaging=None,
        icelltype=icelltype,
        k=k_sw if is_saltwater else k_fw,
    )
    sto = flopy.mf6.ModflowGwfsto(gwf, iconvert=iconvert, ss=ss, sy=sy)
    zeta_file = name + ".zta"
    swi = flopy.mf6.ModflowGwfswi(
        gwf,
        zeta_filerecord=zeta_file,
    )
    chd = flopy.mf6.ModflowGwfchd(
        gwf,
        stress_period_data=[[0, 0, 0, h0], [0, 0, ncol - 1, h0]],
    )
    if not is_saltwater:
        rch = flopy.mf6.ModflowGwfrcha(gwf, recharge=recharge)

    budget_file = name + ".bud"
    head_file = name + ".hds"
    oc = flopy.mf6.ModflowGwfoc(
        gwf,
        budget_filerecord=budget_file,
        head_filerecord=head_file,
        saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
        printrecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
    )

    return gwf


def build_models():
    ws = sim_ws
    sim_name = "mymodel"
    sim = flopy.mf6.MFSimulation(
        sim_name=sim_name,
        sim_ws=ws,
        exe_name=mf6exe,
        memory_print_option="all",
        # continue_=True,
        # print_input=True,
    )

    # transient tdis
    nper = len(perioddata)
    tdis = flopy.mf6.ModflowTdis(sim, nper=nper, perioddata=perioddata)

    ims = flopy.mf6.ModflowIms(
        sim,
        print_option="summary",
        no_ptcrecord=True,
        under_relaxation="DBD",
        under_relaxation_gamma=0.1,
        under_relaxation_theta=0.7,
        under_relaxation_kappa=0.07,
        under_relaxation_momentum=0.0,
        outer_maximum=500,
        inner_maximum=600,
        outer_dvclose=outer_dvclose,
        inner_dvclose=inner_dvclose,
        linear_acceleration="bicgstab",
        preconditioner_levels=7,
        number_orthogonalizations=14,
        preconditioner_drop_tolerance=1e-3,
    )

    gwf_freshwater = build_gwf_model(sim, False)
    gwf_saltwater = build_gwf_model(sim, True)

    swiswi = flopy.mf6.ModflowSwiswi(
        sim,
        print_input=True,
        print_flows=True,
        exgtype="SWI6-SWI6",
        exgmnamea="freshwater",
        exgmnameb="saltwater",
    )
    sim.register_ims_package(ims, [gwf_freshwater.name, gwf_saltwater.name])

    return sim


def plot_output(sim):
    import matplotlib.pyplot as plt

    ws = sim_ws
    gwf = sim.gwf[0]
    x = gwf.modelgrid.xcellcenters.flatten()
    fpth = pl.Path(ws) / f"{gwf.name}.zta"
    hobj = gwf.output.head()
    hsobj = sim.gwf[1].output.head()
    zobj = flopy.utils.HeadFile(fpth, text="zeta")
    times = zobj.times

    ax = plt.subplot(1, 1, 1)
    pxs = flopy.plot.PlotCrossSection(gwf, line={"row": 0}, ax=ax)
    for t in times:
        zeta = zobj.get_data(totim=t).flatten()
        head = hobj.get_data(totim=t).flatten()
        head_salt = hsobj.get_data(totim=t).flatten()
        zeta_calc = -40. * head + 41 * head_salt
        zeta_calc = np.where(zeta_calc > -80, zeta_calc, -80)
        ax.plot(x, zeta, "k-")
        ax.plot(x, zeta_calc, "r--")
    ax.plot(x, head, "b-")
    ax.plot([x.min(), x.max()], [-80, -80], "k-")

    ax.set_ylim(-100, 10.0)
    # plt.savefig(ws / "zeta.png")
    # plt.close("all")


In [ ]:
sim = build_models()
sim.write_simulation()
sim.run_simulation()

In [ ]:
plot_output(sim)

In [ ]:
animator = SwiAnimator(sim=sim)
ani = animator.create()
HTML(ani.to_jshtml())

# Case 8 -- Two-Fluids, Two Stress periods, 2 layers

In [ ]:
ncol = 19 + 2
Lx = 1900.0
dx = Lx / (ncol - 2)
nlay = 2
nrow = 1
delr = np.array([1.] + (ncol - 2) * [dx] + [1.])
#delr = np.array(ncol * [dx])


delc = 1.0
botm = [-40., -80.0]
recharge = {0: 0.0075, 1: 0.0}
k_fw = 10.0
k_sw = 10.0
h0 = 0.0
icelltype = 1
iconvert = 1
top = 10.0
newtonoptions = "NEWTON"
ss = 0.0
sy = 0.2
inner_dvclose = 1.e-4
outer_dvclose = 1.e-3

# perioddata = [(80000., 100, 1.0), (10000., 300, 1.0)] # sorab's original problem
perioddata = [(15000., 100, 1.0), (30000., 100, 1.0)] # this runs


def build_gwf_model(sim, is_saltwater):
    if is_saltwater:
        name = "saltwater"
    else:
        name = "freshwater"

    gwf = flopy.mf6.ModflowGwf(
        sim,
        modelname=name,
        save_flows=True,
        newtonoptions=newtonoptions,
    )
    dis = flopy.mf6.ModflowGwfdis(
        gwf,
        nlay=nlay,
        nrow=nrow,
        ncol=ncol,
        delr=delr,
        delc=delc,
        top=top,
        botm=botm,
    )
    strt = 0.0 if is_saltwater else 0.00
    ic = flopy.mf6.ModflowGwfic(gwf, strt=strt)
    npf = flopy.mf6.ModflowGwfnpf(
        gwf,
        save_specific_discharge=True,
        save_saturation=True,
        # alternative_cell_averaging=None,
        icelltype=icelltype,
        k=k_sw if is_saltwater else k_fw,
    )
    sto = flopy.mf6.ModflowGwfsto(gwf, iconvert=iconvert, ss=ss, sy=sy)
    zeta_file = name + ".zta"
    swi = flopy.mf6.ModflowGwfswi(
        gwf,
        zeta_filerecord=zeta_file,
    )
    chd_spd = [[0, 0, 0, h0], [0, 0, ncol - 1, h0]]
    chd_spd += [[1, 0, 0, h0], [1, 0, ncol - 1, h0]]
    chd = flopy.mf6.ModflowGwfchd(
        gwf,
        stress_period_data=chd_spd,
    )
    if not is_saltwater:
        rch = flopy.mf6.ModflowGwfrcha(gwf, recharge=recharge)

    budget_file = name + ".bud"
    head_file = name + ".hds"
    oc = flopy.mf6.ModflowGwfoc(
        gwf,
        budget_filerecord=budget_file,
        head_filerecord=head_file,
        saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
        printrecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
    )

    return gwf


def build_models():
    ws = sim_ws
    sim_name = "mymodel"
    sim = flopy.mf6.MFSimulation(
        sim_name=sim_name,
        sim_ws=ws,
        exe_name=mf6exe,
        memory_print_option="all",
        # print_input=True,
    )

    # transient tdis
    nper = len(perioddata)
    tdis = flopy.mf6.ModflowTdis(sim, nper=nper, perioddata=perioddata)

    ims = flopy.mf6.ModflowIms(
        sim,
        print_option="summary",
        no_ptcrecord=True,
        under_relaxation="DBD",
        under_relaxation_gamma=0.1,
        under_relaxation_theta=0.7,
        under_relaxation_kappa=0.07,
        under_relaxation_momentum=0.0,
        outer_maximum=500,
        inner_maximum=600,
        outer_dvclose=outer_dvclose,
        inner_dvclose=inner_dvclose,
        linear_acceleration="bicgstab",
        # preconditioner_levels=7,
        # number_orthogonalizations=14,
        # preconditioner_drop_tolerance=1e-3,
    )

    gwf_freshwater = build_gwf_model(sim, False)
    gwf_saltwater = build_gwf_model(sim, True)

    swiswi = flopy.mf6.ModflowSwiswi(
        sim,
        print_input=True,
        print_flows=True,
        exgtype="SWI6-SWI6",
        exgmnamea="freshwater",
        exgmnameb="saltwater",
    )
    sim.register_ims_package(ims, [gwf_freshwater.name, gwf_saltwater.name])

    return sim


def plot_output(sim):
    import matplotlib.pyplot as plt

    ws = sim_ws
    gwf = sim.gwf[0]
    nlay = gwf.dis.nlay.get_data()
    ncol = gwf.dis.ncol.get_data()
    x = gwf.modelgrid.xcellcenters.flatten()
    fpth = pl.Path(ws) / f"{gwf.name}.zta"
    head = gwf.output.head().get_data().flatten().reshape((nlay, ncol))
    head_saltwater = sim.gwf[1].output.head().get_data().flatten().reshape((nlay, ncol))

    zobj = flopy.utils.HeadFile(fpth, text="zeta")
    times = zobj.times

    ax = plt.subplot(1, 1, 1)
    pxs = flopy.plot.PlotCrossSection(gwf, line={"row": 0}, ax=ax)
    for t in times:
        zeta = zobj.get_data(totim=t).flatten().reshape((nlay, ncol))
        for ilay in range(nlay):
            ax.plot(x, zeta[ilay], "k-")
    ax.plot(x, head[0], "b-")
    ax.plot(x, head_saltwater[0], "r-")

    ax.set_ylim(-80, 10.0)
    # plt.savefig(ws / "zeta.png")
    # plt.close("all")


In [ ]:
sim = build_models()
sim.write_simulation()
sim.run_simulation()

In [ ]:
plot_output(sim)

In [ ]:

animator = SwiAnimator(sim=sim)
ani = animator.create()
HTML(ani.to_jshtml())